# Loss ablation -- both phases (water+fat, 2-channel input, 6-class output)

Sibling of `loss_ablation.ipynb`, which screens losses for the water-only
pipeline (1-channel input, 4 chambers). This notebook runs the same screen for
`phase='both'`: `VolumeMRIDataset` stacks the water and fat halves of each
DICOM as a 2-channel image, and the target becomes 6 classes (background + LV
+ RV + LA + RA + EAT) instead of 5. See CLAUDE.md and `utils/data_loading.py`'s
`VolumeMRIDataset` for how the two channels/labels are built.

This is a genuinely new arm, not a re-run of the water ablation with an extra
input channel bolted on: EAT is a new foreground class with its own size and
empty-slice profile, which is exactly what the loss arms below differ in how
they handle. The water ablation's winner is a prior, not an assumption -- this
notebook re-checks it rather than skipping straight to a single run.

Every model here trains from scratch. The first conv layer's shape changed
(1 -> 2 input channels) and the output layer's class count changed (5 -> 6),
so neither the existing water-only nor fat-only checkpoint is weight-compatible
-- there is nothing to warm-start from.

Same three invariants as the water notebook, and they matter identically here:

1. **`split_seed` is frozen at 0** -- identical to `loss_ablation.ipynb`, so the
   val/test patients are the same set and cross-notebook comparisons (see the
   dedicated section near the end) have something to pair on.
2. **The test set is never touched** until a winning arm is chosen.
3. **Selection runs on per-patient macro Dice** (now averaged over all 5
   foreground classes, not 4 -- see the results section for why that makes the
   `all_classes_macro` row here NOT directly comparable to the water notebook's
   `all_classes_macro` row, and what to use instead).

## Current stage: screening, one seed

Same caveat as the water notebook: at `SEEDS = [0]` this can show an arm that's
clearly better/worse/broken, but cannot attribute a small gap (< ~0.01 macro
Dice) to the loss rather than initialisation. Widen to `SEEDS = [0, 1, 2]` and
re-run (finished runs are skipped) before reporting anything.

## Setup

In [9]:
import os

# point this at whichever GPU is free
os.environ["CUDA_VISIBLE_DEVICES"] = "5"
print(f"Targeting GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f8fe6262560, raw_cell="import os

# point this at whichever GPU is free
o.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W2sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Targeting GPU: 5
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f8fe6262f20, execution_count=9 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f8fe6262560, raw_cell="import os

# point this at whichever GPU is free
o.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W2sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [10]:
import sys
sys.path.append('.')

import json
import logging
import numpy as np
import torch
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f949838f9a0, raw_cell="import sys
sys.path.append('.')

import json
impor.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W3sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

Using device: cuda
Tesla V100-PCIE-32GB
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f949838f4f0, execution_count=10 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f949838f9a0, raw_cell="import sys
sys.path.append('.')

import json
impor.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W3sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [11]:
import train
from unet import UNet
from utils import metrics
from utils.data_loading import VolumeMRIDataset
from utils.losses import LOSS_REGISTRY

train.dir_img = Path('./data/imgs/')
train.dir_mask = Path('./data/masks/')
train.dir_checkpoint = Path('./checkpoints/')

print('available arms:', sorted(LOSS_REGISTRY))

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f949838e0e0, raw_cell="import train
from unet import UNet
from utils impo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W4sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

available arms: ['ce', 'dice', 'dice_ce', 'dice_ce_boundary', 'dice_ce_legacy', 'focal_tversky_ce', 'tversky_ce']
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f949838fbb0, execution_count=11 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f949838e0e0, raw_cell="import train
from unet import UNet
from utils impo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W4sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## Experiment configuration

Same three arms as the water ablation, same reasoning for picking them --
see `loss_ablation.ipynb` for the full rationale (`dice_ce` corrected baseline,
`tversky_ce` for the FN/FP asymmetry axis, `dice_ce_boundary` for the
HD95/ASSD axis). `LOSS_REGISTRY` entries are parametric in `n_classes`
(`lambda nc: ...`), so nothing in `utils/losses.py` needed to change for 6
classes instead of 5.

**Cost**: expect a bit more than the water notebook's ~1.1 h / ~1.1 h / ~2.5-3 h
per arm. The 2-channel input barely changes conv cost, but `dice_ce_boundary`'s
CPU-side signed distance transforms (`scipy.ndimage.distance_transform_edt`)
now run once per foreground class -- 5 instead of 4 -- so that arm's per-batch
CPU cost scales up proportionally on top of its existing ~3x multiplier.

`SPLIT_SEED` stays frozen at 0, identical to `loss_ablation.ipynb`.

In [12]:
# --- the ablation ---------------------------------------------------------
PHASE = 'both'
ARMS = [
    'dice_ce',            # corrected baseline -- the reference
    'tversky_ce',         # FN/FP asymmetry axis
    'dice_ce_boundary',   # boundary axis (targets HD95/ASSD)
]
SEED = 0                  # default seed for run_arm(); each run cell below overrides it explicitly
BASELINE_ARM = 'dice_ce'

SEEDS = [0,1,2]                # widen to [0, 1, 2] once ready to report, same as loss_ablation.ipynb

# --- held constant across every arm ---------------------------------------
SPLIT_SEED   = 0        # FROZEN. Identical to loss_ablation.ipynb -- do not change.
EPOCHS       = 40
BATCH_SIZE   = 8
LR           = 1e-5
IMG_SCALE    = 1.0
N_CLASSES    = train.PHASE_N_CLASSES[PHASE]     # 6 for 'both' (bg + LV/RV/LA/RA/EAT)
N_CHANNELS   = train.PHASE_N_CHANNELS[PHASE]    # 2 for 'both' (water, fat)
AMP          = True
AUGMENT      = True
SELECT_ON    = 'macro_dice'
LR_SCHEDULE  = 'poly'   # nnU-Net decay, identical trajectory for every arm
VAL_PERCENT  = 0.15
TEST_PERCENT = 0.15

RUN_PREFIX = 'abl'
run_name_for = lambda arm, seed: f'{RUN_PREFIX}_{arm}_s{seed}'

# train_model prefixes non-water run names with '{phase}_' internally (water
# itself stays unprefixed, for backward compatibility with checkpoints that
# predate `phase` existing at all -- see CLAUDE.md). Any cell here that builds
# a checkpoint path ITSELF, rather than going through train_model, has to
# replicate that same rule or its bookkeeping (already_done, loading CSVs, ...)
# will look in the wrong directory. This is that one rule, defined once --
# identical to loss_ablation_fat.ipynb's run_dir_for().
def run_dir_for(run_name):
    return train.dir_checkpoint / (run_name if PHASE == 'water' else f'{PHASE}_{run_name}')

# rough wall-clock: same base estimate as the water notebook, but the boundary
# arm's distance-transform cost scales with the number of foreground classes (5 vs 4)
est_h = sum((2.6 * (5 / 4) if a == 'dice_ce_boundary' else 1.15) for a in ARMS) * len(SEEDS)
print(f'{len(ARMS)} arms x {len(SEEDS)} seeds {SEEDS}, {EPOCHS} epochs each (~{est_h:.1f} h total):')
for a in ARMS:
    for s in SEEDS:
        print(f'  {run_dir_for(run_name_for(a, s)).name:<28} -> wandb run of the same name')
assert set(ARMS) <= set(LOSS_REGISTRY), set(ARMS) - set(LOSS_REGISTRY)
assert BASELINE_ARM in ARMS

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f94983cd390, raw_cell="# --- the ablation -------------------------------.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#W6sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

3 arms x 3 seeds [0, 1, 2], 40 epochs each (~16.6 h total):
  both_abl_dice_ce_s0          -> wandb run of the same name
  both_abl_dice_ce_s1          -> wandb run of the same name
  both_abl_dice_ce_s2          -> wandb run of the same name
  both_abl_tversky_ce_s0       -> wandb run of the same name
  both_abl_tversky_ce_s1       -> wandb run of the same name
  both_abl_tversky_ce_s2       -> wandb run of the same name
  both_abl_dice_ce_boundary_s0 -> wandb run of the same name
  both_abl_dice_ce_boundary_s1 -> wandb run of the same name
  both_abl_dice_ce_boundary_s2 -> wandb run of the same name
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f94983cc610, execution_count=12 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f94983cd390, raw_cell="# --- the ablation -------------------------------.." store_history

ConnectionResetError: Connection lost

## Dataset

Built **once** and passed into every run via `dataset=`. `VolumeMRIDataset(..., phase='both')`
stacks the water and fat halves of each of the 49 DICOM volumes as a 2-channel
image and keeps all 6 mask labels; this is cached to `data/preprocessed_cache/`
under a `_both`-suffixed filename, distinct from the water-only and fat-only
caches, so the first run here reprocesses every patient once (one-time cost).
Rebuilding the dataset per arm would repeat that work 3x for no reason, and
would also thrash the in-memory LRU cache.

In [13]:
dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=IMG_SCALE, phase=PHASE)
print(f'{len(dataset.mask_file_for)} patients, {len(dataset.index)} slices')
print(f'mask values: {dataset.mask_values}')

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f949840b100, raw_cell="dataset = VolumeMRIDataset(train.dir_img, train.di.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X11sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

INFO: Found 50 patients, 4748 total slices
INFO: Scanning mask files to determine unique values...
INFO: Unique mask values: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]


50 patients, 4748 slices
mask values: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f94984093c0, execution_count=13 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f949840b100, raw_cell="dataset = VolumeMRIDataset(train.dir_img, train.di.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X11sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

### Confirm the split before spending any GPU time

This is the assertion that protects the whole notebook. If these patient lists
are not identical for every arm, `compare_runs` has nothing to pair on.

In [14]:
train_idx, val_idx, test_idx = train.split_patients(
    dataset, val_percent=VAL_PERCENT, test_percent=TEST_PERCENT, seed=SPLIT_SEED
)

val_patients  = sorted({dataset.index[i][0] for i in val_idx})
test_patients = sorted({dataset.index[i][0] for i in test_idx})

print('val patients :', val_patients)
print('test patients:', test_patients, '  <- not scored until the final section')
print(f'slices -> train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}')

# disjoint at the PATIENT level, not just the slice level
pat = lambda idxs: {dataset.index[i][0] for i in idxs}
assert not (pat(train_idx) & pat(val_idx))
assert not (pat(train_idx) & pat(test_idx))
assert not (pat(val_idx) & pat(test_idx))
print('OK: train / val / test disjoint by patient')

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f949838e290, raw_cell="train_idx, val_idx, test_idx = train.split_patient.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X13sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

INFO: Patient-level split of 50 patients / 4748 slices (seed 0):
        train: 34 patients,  3232 slices
        val:    8 patients,   742 slices  ['CADRE_1382_second', 'CADRE_1395_second', 'CADRE_1671', 'CADRE_1855', 'CADRE_1857', 'CADRE_1861', 'CADRE_1872', 'CADRE_1875']
        test:   8 patients,   774 slices  ['CADRE_1113', 'CADRE_1116_first', 'CADRE_1523', 'CADRE_1532', 'CADRE_1743', 'CADRE_1744', 'CADRE_1859', 'CADRE_1867']


val patients : ['CADRE_1382_second', 'CADRE_1395_second', 'CADRE_1671', 'CADRE_1855', 'CADRE_1857', 'CADRE_1861', 'CADRE_1872', 'CADRE_1875']
test patients: ['CADRE_1113', 'CADRE_1116_first', 'CADRE_1523', 'CADRE_1532', 'CADRE_1743', 'CADRE_1744', 'CADRE_1859', 'CADRE_1867']   <- not scored until the final section
slices -> train 3232 / val 742 / test 774
OK: train / val / test disjoint by patient


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f949838e050, execution_count=14 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f949838e290, raw_cell="train_idx, val_idx, test_idx = train.split_patient.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X13sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

## Run the arms

One cell per arm, so each can be run, watched, and re-run on its own. They share
`run_arm()` below rather than repeating the twenty-odd `train_model` arguments
three times -- if those drifted apart between cells the arms would stop being
comparable, which is the one thing this notebook exists to prevent.

Each call produces **its own wandb run**, named after the arm
(`abl_both_dice_ce_s0`, ...), tagged `both` (from `phase='both'`). The chosen
loss is in that run's config, so the three are filterable by `config.loss` in
the UI.

Each call also writes, into `checkpoints/<run_name>/`:

| file | contents |
|---|---|
| `val_metrics_per_patient.csv` | one row per (patient, class) -- Dice, HD95 mm, ASSD mm, voxel counts, now for 5 classes (LV/RV/LA/RA/EAT) |
| `val_metrics_summary.csv` | mean / SD / median / IQR across the val patients, per class, plus the `all_classes_macro` row (now averaged over 5 classes, not 4 -- see the results section) |
| `run_config.json` | every hyperparameter, the arm name, `phase: 'both'`, the patient lists, per-epoch history |
| `best.pth`, `checkpoint_epoch<N>.pth` | weights |

Re-running a cell whose run already finished skips it and re-loads from disk.
Delete `checkpoints/<run_name>/` to force a genuine re-run.

The model is rebuilt inside `run_arm` after `set_seed` -- `UNet(...)` draws its
initial weights from the torch RNG, so seeding afterwards would leave the init
random, and reusing a model object across arms would silently start one arm from
another's trained weights.

In [15]:
import gc
import time

results = {}      # arm -> the dict train_model returned (or the reloaded config)


def already_done(run_name):
    cfg = run_dir_for(run_name) / 'run_config.json'
    if not cfg.exists():
        return False
    try:
        return 'finished' in json.loads(cfg.read_text())
    except json.JSONDecodeError:
        return False


def run_arm(arm, seed=SEED, force=False):
    """Train one ablation arm. Every argument except `loss` is shared, which is
    what makes the arms comparable -- do not vary them per cell."""
    run_name = run_name_for(arm, seed)
    run_dir = run_dir_for(run_name)

    if already_done(run_name) and not force:
        cfg = json.loads((run_dir / 'run_config.json').read_text())
        print(f'{run_name} already finished (best epoch {cfg["best_epoch"]}, '
              f'macro Dice {cfg.get("best_val_macro_dice", float("nan")):.4f}).')
        print(f'Pass force=True or delete {run_dir} to re-run.')
        results[arm] = cfg
        return cfg

    print(f'===== {run_name} =====')
    t0 = time.time()

    train.set_seed(seed)          # BEFORE the model: UNet() consumes the torch RNG
    model = UNet(n_channels=N_CHANNELS, n_classes=N_CLASSES, bilinear=False)
    model = model.to(memory_format=torch.channels_last).to(device=device)

    res = train.train_model(
        model=model,
        device=device,
        dataset=dataset,          # built once above, reused by every arm
        loss=arm,                 # <-- the only thing that varies between arms
        phase=PHASE,              # 'both' -- dataset is passed explicitly so this
                                   # is the only way train_model learns the phase
                                   # (needed for class_names lookups + run-name prefixing)
        seed=seed,
        split_seed=SPLIT_SEED,    # frozen, identical to loss_ablation.ipynb
        select_on=SELECT_ON,
        lr_schedule=LR_SCHEDULE,  # deterministic: same LR curve for every arm
        run_name=run_name,        # also becomes the wandb run name
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        img_scale=IMG_SCALE,
        amp=AMP,
        augment=AUGMENT,
        val_percent=VAL_PERCENT,
        test_percent=TEST_PERCENT,
        eval_test=False,          # the test set stays sealed
    )
    results[arm] = res

    print(f'\n{run_name}: best macro Dice {res["best_val_macro_dice"]:.4f} '
          f'@ epoch {res["best_epoch"]}  ({(time.time() - t0) / 60:.1f} min)')
    print(f'mean/SD across val patients -> {run_dir / "val_metrics_summary.csv"}')
    if 'val' in res.get('per_patient_metrics', {}):
        print(metrics.format_summary(res['per_patient_metrics']['val']['summary']))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return res


print('run_arm() ready. Run the three cells below one at a time.')

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f94982acc40, raw_cell="import gc
import time

results = {}      # arm -> .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X15sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

run_arm() ready. Run the three cells below one at a time.
Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f94982af220, execution_count=15 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f94982acc40, raw_cell="import gc
import time

results = {}      # arm -> .." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X15sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

### Arm 1 / 3 — `dice_ce` (baseline, ~1.1 h)

Corrected Dice + CE at 1:1. Per-class soft Dice, background excluded,
numerator/denominator pooled over the batch. Everything else is measured
against this.

In [16]:
res_dice_ce = {seed: run_arm('dice_ce', seed=seed) for seed in SEEDS}

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f949812d450, raw_cell="res_dice_ce = {seed: run_arm('dice_ce', seed=seed).." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X20sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


ConnectionResetError: Connection lost

INFO: Seeded RNGs with 0


===== abl_dice_ce_s0 =====


INFO: Seeded RNGs with 0
INFO: Patient-level split of 50 patients / 4748 slices (seed 0):
        train: 34 patients,  3232 slices
        val:    8 patients,   742 slices  ['CADRE_1382_second', 'CADRE_1395_second', 'CADRE_1671', 'CADRE_1855', 'CADRE_1857', 'CADRE_1861', 'CADRE_1872', 'CADRE_1875']
        test:   8 patients,   774 slices  ['CADRE_1113', 'CADRE_1116_first', 'CADRE_1523', 'CADRE_1532', 'CADRE_1743', 'CADRE_1744', 'CADRE_1859', 'CADRE_1867']
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Exception in thread wandb-ServiceFinalizer:
Traceback (most recent call last):
  File "/hpc/jgeo610/Virtual_ENV/unet_env/lib/python3.10/site-packages/wandb/sdk/lib/asyncio_manager.py", line 195, in fn_wrap_exceptions
    await fn()
  File "/hpc/jgeo610/Virtual_ENV/unet_env/lib/python3.10/site-packages/wandb/sdk/lib/service/service_client.py", line 42, in publish
    await self._send_server_request(request)
  File "/hpc/jgeo610/Virtual_ENV/unet_env/lib/python3.10/site-packages/wandb/sdk/lib/service/service_client.py", line 75, in _send_server_request
    raise self._broken_exc.with_traceback(self._broken_tb)
  File "/hpc/jgeo610/Virtual_ENV/unet_env/lib/python3.10/site-packages/wandb/sdk/lib/service/service_client.py", line 84, in _send_server_request
    await self._drain_writer()
  File "/hpc/jgeo610/Virtual_ENV/unet_env/lib/python3.10/site-packages/wandb/sdk/lib/service/service_client.py", line 92, in _drain_writer
    await self._writer.drain()
  File "/usr/local/fsl/lib/python3.10/

ConnectionResetError: Connection lost

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f8feb8e7700>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f949812d720, execution_count=16 error_before_exec=None error_in_exec=Connection lost info=<ExecutionInfo object at 7f949812d450, raw_cell="res_dice_ce = {seed: run_arm('dice_ce', seed=seed).." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Buoa-hpc/hpc/jgeo610/heart_segmentation/both_phase_ablation.ipynb#X20sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

### Arm 2 / 3 — `tversky_ce` (~1.1 h)

Tversky (α=0.3, β=0.7) + CE. β>α makes a false negative cost more than a false
positive, which is the standard lever for structures the model under-segments.
If it helps anywhere here it should show up in LA/RA, possibly at some cost to
LV precision.

In [ ]:
res_tversky_ce = {seed: run_arm('tversky_ce', seed=seed) for seed in SEEDS}

### Arm 3 / 3 — `dice_ce_boundary` (~2.5–3 h, the slow one)

Dice + CE with a Kervadec boundary term, region weight ramping 1.0 → 0.01 and
the boundary weight rising to meet it. The only arm aimed at HD95/ASSD rather
than overlap.

**Run this one last.** Its signed distance transforms run on CPU, serially, in
the main process — measured at ~540 ms/batch at 432×432, roughly 3× the wall
clock of the other two. If Dice barely moves but HD95 drops, that is the
expected result, not a failure.

In [ ]:
res_dice_ce_boundary = {seed: run_arm('dice_ce_boundary', seed=seed) for seed in SEEDS}

## Results

### Per-run summary

Read back from disk rather than from the in-memory `results` dicts, so this
section works on a fresh kernel and reflects exactly what was written.

In [ ]:
def load_run(run_name):
    d = run_dir_for(run_name)
    cfg = json.loads((d / 'run_config.json').read_text())
    csv_path = d / 'val_metrics_per_patient.csv'
    rows = metrics.load_per_patient(csv_path) if csv_path.exists() else []
    return cfg, rows


# dice, precision, recall, hd95_mm, assd_mm -- metrics absent from a CSV are
# dropped by summarise() rather than raising, so pre-fix runs still load.
PER_PATIENT_METRICS = metrics.DEFAULT_METRICS


def arm_rows(arm):
    """Per-patient rows for one arm, averaged over whatever seeds have finished.

    Averaging per (patient, class) FIRST matters once SEEDS has more than one
    entry: the same 7 patients scored 3 times are not 21 independent
    observations. With a single seed this is just a pass-through.
    """
    by_key, found = {}, []
    for seed in SEEDS:
        path = (run_dir_for(run_name_for(arm, seed))
                / 'val_metrics_per_patient.csv')
        if not path.exists():
            continue
        found.append(seed)
        for r in metrics.load_per_patient(path):
            by_key.setdefault((r['patient_id'], r['class_name']), []).append(r)

    out = []
    for (pid, cname), rs in sorted(by_key.items()):
        row = {'patient_id': pid, 'class_name': cname, 'cls': rs[0]['cls']}
        for m in PER_PATIENT_METRICS + ('voxel_ml',):
            if m not in rs[0]:
                continue
            vals = [r[m] for r in rs if np.isfinite(r[m])]
            row[m] = float(np.mean(vals)) if vals else float('nan')
        for m in ('gt_voxels', 'pred_voxels'):
            row[m] = int(np.mean([r[m] for r in rs]))
        row['missed'] = any(r['missed'] for r in rs)
        row['n_seeds'] = len(rs)
        out.append(row)
    return out, set(found)


# --- one line per finished run --------------------------------------------
table = []
for arm in ARMS:
    for seed in SEEDS:
        run_name = run_name_for(arm, seed)
        if not (run_dir_for(run_name) / 'run_config.json').exists():
            continue
        cfg, rows = load_run(run_name)
        macro = [r for r in metrics.summarise(rows) if r['cls'] == -1]
        m = macro[0] if macro else {}
        table.append({
            'arm': arm, 'seed': seed,
            'best_epoch': cfg.get('best_epoch'),
            'macro_dice': m.get('dice_mean', float('nan')),
            'macro_precision': m.get('precision_mean', float('nan')),
            'macro_recall': m.get('recall_mean', float('nan')),
            'macro_hd95': m.get('hd95_mm_mean', float('nan')),
            'macro_assd': m.get('assd_mm_mean', float('nan')),
            'n_missed': m.get('n_missed', 0),
            'split_seed': cfg.get('split_seed'),
            'loss': cfg.get('loss'),
        })

# every arm must have used the same split, or nothing below is comparable
split_seeds = {t['split_seed'] for t in table}
assert len(split_seeds) <= 1, f'runs used different split_seeds: {split_seeds}'
# and the recorded loss must match the arm the run was filed under
for t in table:
    assert t['loss'] in (None, t['arm']), f'{t["arm"]} run recorded loss={t["loss"]}'

# precision < recall means the arm over-segments (claiming voxels it shouldn't);
# recall < precision means it under-segments. Dice is their harmonic mean and
# cannot tell you which.
print(f'{"arm":<18}{"seed":>5}{"epoch":>7}{"Dice":>9}{"Prec":>9}{"Rec":>9}'
      f'{"HD95 mm":>10}{"ASSD mm":>10}{"missed":>8}')
for t in table:
    print(f'{t["arm"]:<18}{t["seed"]:>5}{t["best_epoch"]:>7}'
          f'{t["macro_dice"]:>9.4f}{t["macro_precision"]:>9.4f}{t["macro_recall"]:>9.4f}'
          f'{t["macro_hd95"]:>10.2f}{t["macro_assd"]:>10.2f}{t["n_missed"]:>8}')

### Combined mean ± SD table, written to one CSV

Each run already wrote its own `val_metrics_summary.csv` — mean, SD, median and
IQR across the 7 validation patients, per chamber plus the macro row. That is
the per-run version of what you want.

This cell stacks all three into a single `ablation_val_summary.csv` with an
`arm` column, so the comparison lives in one file instead of three. Statistics
are recomputed from the per-patient CSVs rather than concatenated from the
summaries, so a future multi-seed run averages per patient first and the file
stays correct without changes.

`n` is how many patients contributed to each statistic and `n_missed` how many
had the structure present but predicted empty — an HD95 mean sitting on fewer
than 7 patients is only interpretable next to those two columns.

In [ ]:
ABLATION_DIR = train.dir_checkpoint / f'{PHASE}_{RUN_PREFIX}_summary'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

combined = []
for arm in ARMS:
    rows, seeds_found = arm_rows(arm)          # defined in the cell above
    if not rows:
        print(f'{arm}: not run yet, skipped')
        continue
    for r in metrics.summarise(rows):
        out = {
            'arm': arm,
            'seeds': ','.join(str(s) for s in sorted(seeds_found)),
            'split_seed': SPLIT_SEED,
            'class_name': r['class_name'],
            'cls': r['cls'],
            'n_patients': r['n_patients'],
            'n_missed': r['n_missed'],
        }
        for m in PER_PATIENT_METRICS:          # dice, precision, recall, hd95_mm, assd_mm
            if f'{m}_mean' not in r:
                continue
            for stat in ('mean', 'sd', 'median', 'n'):
                out[f'{m}_{stat}'] = r[f'{m}_{stat}']
        combined.append(out)

csv_path = ABLATION_DIR / 'ablation_val_summary.csv'
metrics.write_csv(combined, csv_path)
print(f'wrote {len(combined)} rows to {csv_path}\n')

print(f'{"arm":<20}{"class":<20}{"Dice":>17}{"Prec":>17}{"Rec":>17}'
      f'{"HD95 mm":>17}{"n":>4}{"missed":>8}')
for r in combined:
    cells = ''
    for m, dp in [('dice', 4), ('precision', 4), ('recall', 4), ('hd95_mm', 2)]:
        cells += (f'{r[f"{m}_mean"]:>10.{dp}f}+/-{r[f"{m}_sd"]:<4.2f}'
                  if f'{m}_mean' in r else f'{"-":>17}')
    print(f'{r["arm"]:<20}{r["class_name"]:<20}{cells}'
          f'{r["n_patients"]:>4}{r["n_missed"]:>8}')

### Per-class, side by side

This is the table to read at the screening stage. Three metrics, five classes
(LV/RV/LA/RA/EAT) plus the macro row, one column per arm, `mean +/- SD` across
the validation patients (SD is `nan` when only one seed has run and a class
has one value).

What to look for:

- **Dice** -- does any arm move the macro row by more than ~0.01? Smaller than
  that is inside one-seed noise.
- **HD95 / ASSD** -- this is where `dice_ce_boundary` should earn its extra
  runtime, if it earns it at all.
- **LA and RA specifically** -- `tversky_ce`'s beta>alpha is aimed at recall on
  the thinner-walled atria; the same logic could plausibly help EAT too, since
  epicardial fat is a thin, easily-under-segmented rim.
- **`n_missed` must be 0.** Any other value means a class was predicted empty
  for some patient, and the HD95/ASSD means above it are computed on the
  remaining patients only.

In [ ]:
# side-by-side, per chamber, across arms (arm_rows is defined above)
summaries = {}
for arm in ARMS:
    rows, _seeds = arm_rows(arm)
    if rows:
        summaries[arm] = {r['class_name']: r for r in metrics.summarise(rows)}

ORDER = ['LV', 'RV', 'LA', 'RA', 'EAT', 'all_classes_macro']
have = [a for a in ARMS if a in summaries]

for metric, unit, better in [('dice', '', 'higher'),
                             ('hd95_mm', ' mm', 'lower'),
                             ('assd_mm', ' mm', 'lower')]:
    print(f'\n=== {metric}{unit}  ({better} is better)   '
          f'mean +/- SD over {len(val_patients)} val patients ===')
    print(f'{"":<20}' + ''.join(f'{a:>24}' for a in have))
    for chamber in ORDER:
        cells = []
        for a in have:
            r = summaries[a].get(chamber)
            if r is None:
                cells.append(f'{"-":>24}')
            else:
                cells.append(f'{r[f"{metric}_mean"]:>15.4f} +/- {r[f"{metric}_sd"]:<6.3f}'
                             if metric == 'dice' else
                             f'{r[f"{metric}_mean"]:>15.2f} +/- {r[f"{metric}_sd"]:<6.2f}')
        print(f'{chamber:<20}' + ''.join(cells))

print('\nn_missed (GT present, prediction empty -- must be 0 to trust the distance rows):')
for a in have:
    print(f'  {a:<20}{summaries[a]["all_classes_macro"]["n_missed"]}')

## Paired comparison against the baseline — optional at this stage

**At `SEEDS = [0]` this section is not informative.** It will run, and it will
print p-values, but with one seed per arm any difference it finds is part loss
and part initialisation, and there is no way to tell which. Skip it while
screening; the tables and figures above are what you want.

It becomes the right tool once `SEEDS = [0, 1, 2]`. Kept here so that widening
seeds requires no new code.

`compare_runs` pairs on `(patient_id, class_name)` and runs a paired Wilcoxon
signed-rank test. Paired is the right test: every arm scored the same 7
patients, and between-patient variance dominates everything else at this n.

**On combining seeds.** Each patient is scored once per seed. Pooling those as
if they were independent observations would be wrong — they are the same 7
patients. So each patient's metric is averaged across seeds first, giving one
value per (patient, class) per arm, and the pairing is done on those.

Note the sample-size floor `compare_runs` applies automatically: the exact
two-sided Wilcoxon p cannot go below 2/2⁷ = 0.0156 at n=7, so the 12
per-chamber tests can never survive Holm correction (0.0156 × 12 = 0.19). The
macro row is the single pre-specified primary endpoint; per-chamber rows are
descriptive.

In [ ]:
SEED_MEAN_DIR = train.dir_checkpoint / f'{PHASE}_{RUN_PREFIX}_seedmean'
SEED_MEAN_DIR.mkdir(parents=True, exist_ok=True)

arm_seeds = {}      # which seeds actually contributed to each arm

seed_mean_paths = {}
for arm in ARMS:
    rows, arm_seeds[arm] = arm_rows(arm)      # defined in the results section
    path = SEED_MEAN_DIR / f'{arm}_val_metrics_per_patient.csv'
    metrics.write_csv(rows, path)             # compare_runs reads CSVs, not objects
    seed_mean_paths[arm] = path

print('seed-averaged CSVs written to', SEED_MEAN_DIR)
for arm in ARMS:
    print(f'  {arm:<18} seeds {sorted(arm_seeds[arm]) or "-- none run --"}')

In [ ]:
baseline_csv = seed_mean_paths[BASELINE_ARM]

for arm in ARMS:
    if arm == BASELINE_ARM:
        continue
    if not arm_seeds[arm]:
        print(f'\n== {arm}: not run yet, skipping')
        continue

    # Unbalanced seeds are a silent confound: if the baseline carries 3 seeds and
    # this arm carries 1, the difference between them is part loss and part
    # which init each happened to get. Compare only on the seeds both have.
    if arm_seeds[arm] != arm_seeds[BASELINE_ARM]:
        shared = sorted(arm_seeds[arm] & arm_seeds[BASELINE_ARM])
        print(f'\n== SKIPPING {arm}: seeds {sorted(arm_seeds[arm])} vs baseline '
              f'{sorted(arm_seeds[BASELINE_ARM])}. Run the missing ones, or set '
              f'SEEDS = {shared} and re-run the seed-averaging cell.')
        continue

    print(f'\n===== {BASELINE_ARM}  vs  {arm}  (validation, n={len(val_patients)} '
          f'patients, seeds {sorted(arm_seeds[arm])}) =====')
    comparison = metrics.compare_runs(
        baseline_csv, seed_mean_paths[arm],
        label_a=BASELINE_ARM, label_b=arm,
    )
    print(metrics.format_comparison(comparison, label_a=BASELINE_ARM, label_b=arm))

## Compare against the existing single-phase models

Two separate questions, both answerable with `metrics.compare_runs` on raw
per-patient CSVs (not the pre-computed `val_metrics_summary.csv` files -- those
each carry an `all_classes_macro` row averaged over however many classes *that*
run has, so a 5-class `'both'` macro is not directly comparable to a 4-class
water-only macro by simply reading the two numbers side by side).

`compare_runs` sidesteps that by pairing on `(patient_id, class_name)`: any
class present in one CSV but not the other (e.g. EAT, absent from the water-only
run) simply has no pair and drops out of both the per-chamber tests and the
macro row automatically -- so `compare_runs(water_csv, both_csv)`'s macro row
*is* already the fair chambers-only comparison, with no separate filtering step
needed.

1. **Chambers**: does adding the fat channel/class help or hurt LV/RV/LA/RA,
   relative to the existing water-only model?
2. **EAT**: does joint training help or hurt EAT segmentation, relative to the
   existing dedicated `phase='fat'` model? This only pairs correctly because
   `BOTH_CLASS_NAMES` names label 5 `'EAT'` (matching `PHASE_CLASS_NAMES['fat']`),
   not `'Fat'` -- if that ever drifts, this section silently pairs nothing.

`WATER_BASELINE_CSV` / `FAT_BASELINE_CSV` below default to the existing
`abl_dice_ce_s0` (water ablation baseline) and `fat_baseline_vs_aug_seedmean`
(seed-averaged fat baseline) runs found in `checkpoints/` -- **check these are
still the runs you consider "current best" before trusting the comparison**,
and swap in a different `val_metrics_per_patient.csv` path if not. Both
comparisons require `split_seed=0` on the baseline runs, same as this notebook,
or there are no shared patients to pair on.

In [ ]:
WATER_BASELINE_CSV = train.dir_checkpoint / 'abl_dice_ce_s0' / 'val_metrics_per_patient.csv'
FAT_BASELINE_CSV = train.dir_checkpoint / 'fat_baseline_vs_aug_seedmean' / 'baseline_val_metrics_per_patient.csv'
COMPARE_ARM = BASELINE_ARM   # which 'both'-phase arm to compare -- defaults to this notebook's baseline

both_csv = seed_mean_paths.get(COMPARE_ARM) if 'seed_mean_paths' in dir() else None
if both_csv is None or not arm_seeds.get(COMPARE_ARM):
    print(f'{COMPARE_ARM} has no seed-averaged CSV yet -- run the seed-averaging cell above first.')
elif not WATER_BASELINE_CSV.exists():
    print(f'{WATER_BASELINE_CSV} not found -- update WATER_BASELINE_CSV to point at your current water baseline run.')
else:
    print(f'===== water-only baseline  vs  both[{COMPARE_ARM}]  (chambers, shared classes only) =====')
    water_cmp = metrics.compare_runs(
        WATER_BASELINE_CSV, both_csv,
        label_a='water_baseline', label_b=f'both_{COMPARE_ARM}',
    )
    print(metrics.format_comparison(water_cmp, label_a='water_baseline', label_b=f'both_{COMPARE_ARM}'))

if both_csv is not None and arm_seeds.get(COMPARE_ARM):
    if not FAT_BASELINE_CSV.exists():
        print(f'\n{FAT_BASELINE_CSV} not found -- update FAT_BASELINE_CSV to point at your current fat baseline run.')
    else:
        print(f'\n===== fat-only baseline  vs  both[{COMPARE_ARM}]  (EAT only, shared classes only) =====')
        fat_cmp = metrics.compare_runs(
            FAT_BASELINE_CSV, both_csv,
            label_a='fat_baseline', label_b=f'both_{COMPARE_ARM}',
        )
        print(metrics.format_comparison(fat_cmp, label_a='fat_baseline', label_b=f'both_{COMPARE_ARM}'))

## Figures

Boxplots across arms, per chamber. A mean ± SD over 7 patients hides the single
patient that a loss fails on; the box shows it.

In [ ]:
import matplotlib.pyplot as plt

ORDER = ['LV', 'RV', 'LA', 'RA', 'EAT']
METRIC = 'dice'          # 'dice' | 'precision' | 'recall' | 'hd95_mm' | 'assd_mm'

# Built from arm_rows() rather than seed_mean_paths, so this figure does not
# depend on having run the paired-comparison section (which is skippable at one seed).
loaded = {arm: arm_rows(arm)[0] for arm in ARMS}
loaded = {a: rs for a, rs in loaded.items() if rs}

# fixed generator: the jitter is cosmetic, but an unseeded one redraws the
# figure differently every execution, which is not what you want in a report
jitter = np.random.default_rng(0)
arms_present = [a for a in ARMS if a in loaded]

fig, axes = plt.subplots(1, len(ORDER), figsize=(4.3 * len(ORDER), 4.8), sharey=True)
for ax, chamber in zip(axes, ORDER):
    data = [[r[METRIC] for r in loaded[a]
             if r['class_name'] == chamber and np.isfinite(r[METRIC])]
            for a in arms_present]

    ax.boxplot(data, tick_labels=arms_present, showmeans=True, widths=0.6)

    # n = 7 patients, so the "box" spans three or four points and the whiskers
    # describe a distribution that was never really measured. The individual
    # patients ARE the data -- always show them at this sample size.
    for i, vals in enumerate(data, start=1):
        ax.scatter(jitter.normal(i, 0.045, len(vals)), vals,
                   s=26, alpha=0.75, zorder=3, edgecolor='none')

    n = max((len(v) for v in data), default=0)
    ax.set_title(f'{chamber}  (n={n} patients)')
    ax.tick_params(axis='x', rotation=90)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel(METRIC)
fig.suptitle(f'{METRIC} by chamber and loss (validation, one point per patient)')
fig.tight_layout()
plt.show()

In [ ]:
# Training curves: is any arm still improving at EPOCHS, or diverging?
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for arm in ARMS:
    for seed in SEEDS:
        cfg_path = run_dir_for(run_name_for(arm, seed)) / 'run_config.json'
        if not cfg_path.exists():
            continue
        hist = json.loads(cfg_path.read_text()).get('history', [])
        if not hist:
            continue
        ep = [h['epoch'] for h in hist]
        label = arm if seed == SEEDS[0] else None
        axes[0].plot(ep, [h['train_loss'] for h in hist], alpha=0.7, label=label)
        axes[1].plot(ep, [h.get('val_macro_dice', float('nan')) for h in hist],
                     alpha=0.7, label=label)

axes[0].set_title('train loss')
axes[0].set_xlabel('epoch')
# arms use different losses, so the absolute values are NOT comparable between
# curves -- only the shape of each curve is informative
axes[0].set_yscale('log')
axes[1].set_title('validation macro Dice (per-patient)')
axes[1].set_xlabel('epoch')
for a in axes:
    a.grid(alpha=0.3)
    a.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Final: score the winner on the test set, once

Everything above used validation only. Run this **after** the winning arm is
decided, and run it once.

Two ways to get the number, and they are not equivalent:

- `RETRAIN = False` reloads `best.pth` from an existing ablation run and scores
  test with it. Cheap and correct: those weights were selected on validation and
  have never seen test.
- `RETRAIN = True` trains a fresh model on train+val with the winning loss. Only
  do this if you intend to report a model trained on more data — and if you do,
  the val patients are no longer available as a selection signal.

Report the result as a final number. Do not go back and change an arm because
of it; that is what the validation section was for.

In [ ]:
WINNING_ARM = BASELINE_ARM     # <- set this from the results above
WINNING_SEED = SEEDS[0]
RUN_FINAL_TEST = False         # <- flip to True deliberately, once

if RUN_FINAL_TEST:
    run_dir = run_dir_for(run_name_for(WINNING_ARM, WINNING_SEED))
    model = UNet(n_channels=N_CHANNELS, n_classes=N_CLASSES, bilinear=False)
    model = model.to(memory_format=torch.channels_last).to(device=device)

    state_dict = torch.load(run_dir / 'best.pth', map_location=device)
    state_dict.pop('mask_values', None)      # injected by train.py, not a real weight
    model.load_state_dict(state_dict)
    model.eval()

    rows, summary = metrics.report(
        model, dataset, test_idx, device,
        n_classes=N_CLASSES,
        out_dir=run_dir, split_name='test',
        class_names=train.PHASE_CLASS_NAMES[PHASE],
        amp=AMP, batch_size=BATCH_SIZE,
    )
    print(f'FINAL -- {WINNING_ARM} (seed {WINNING_SEED}) on {len(test_patients)} '
          f'held-out test patients:\n')
    print(metrics.format_summary(summary))
    print('\nCSVs written to', run_dir)
else:
    print('Test evaluation is gated. Set RUN_FINAL_TEST = True once the loss is chosen.')